In [1]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver

In [2]:
load_dotenv()

llm= ChatGroq(model='openai/gpt-oss-20b')

In [3]:
class JokeState(TypedDict):
    topic:str
    joke:str
    explanation:str

In [4]:
def generate_joke(state: JokeState):
    prompt = f'generate a joke on the topic: {state["topic"]}'
    response = llm.invoke(prompt).content
    return {"joke": response}

In [5]:
def generate_explanation(state: JokeState):
    prompt = f'write a exlanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content
    return {"explanation": response}

In [6]:
graph = StateGraph(JokeState)

graph.add_node("generate_joke", generate_joke)
graph.add_node("generate_explanation", generate_explanation)

graph.add_edge(START, "generate_joke")
graph.add_edge("generate_joke", "generate_explanation")
graph.add_edge("generate_explanation", END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [7]:
config1 = {'configurable':{'thread_id':'1'}}
workflow.invoke({'topic':'pizza'},config=config1)

{'topic': 'pizza',
 'joke': 'Why did the pizza apply for a job?  \n\nBecause it wanted to get a *crust* of a living! 🍕😄',
 'explanation': '**Why the joke works – a quick breakdown**\n\n| Element | What it is | Why it’s funny |\n|---------|------------|----------------|\n| **The set‑up** | “Why did the pizza apply for a job?” | It instantly anthropomorphises the pizza—imagine a pizza with a résumé, a cover letter, a “resume” that’s actually a slice of dough. The absurdity of a food item applying for employment is a classic setup for a pun. |\n| **The punch‑line** | “Because it wanted to get a *crust* of a living!” | The humor hinges on the word *crust* sounding almost identical to *cost*. |\n| **Double meaning** | 1. **Crust** – the bread‑like base that holds a pizza together. 2. **Cost** – the amount of money needed to cover living expenses (“cost of living”). | The joke replaces the ordinary “cost” with the literal pizza element “crust,” turning a mundane phrase into a pizza‑centric p

In [9]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza apply for a job?  \n\nBecause it wanted to get a *crust* of a living! 🍕😄', 'explanation': '**Why the joke works – a quick breakdown**\n\n| Element | What it is | Why it’s funny |\n|---------|------------|----------------|\n| **The set‑up** | “Why did the pizza apply for a job?” | It instantly anthropomorphises the pizza—imagine a pizza with a résumé, a cover letter, a “resume” that’s actually a slice of dough. The absurdity of a food item applying for employment is a classic setup for a pun. |\n| **The punch‑line** | “Because it wanted to get a *crust* of a living!” | The humor hinges on the word *crust* sounding almost identical to *cost*. |\n| **Double meaning** | 1. **Crust** – the bread‑like base that holds a pizza together. 2. **Cost** – the amount of money needed to cover living expenses (“cost of living”). | The joke replaces the ordinary “cost” with the literal pizza element “crust,” turning a mundane phrase int

In [ ]:
list[workflow.get_state_history(config1)]

list[<generator object Pregel.get_state_history at 0x000002936C57D2A0>]
